# Day 8 Assignment — Hands-On with Spark Streaming
**PySpark Structured Streaming: Twitter/X Hashtag Popularity**

Adapted to the step-by-step SparkSession/DataFrame/SQL style of the supplied M4 flight-delay practice notebook. This notebook uses simulated incoming tweets by default, so no X API credentials are needed. It also includes an optional socket-based live text source.

## Step 1 — Install and configure PySpark
Run in Google Colab. If running locally, install PySpark 3.5.1 and Java 11 separately. Restart the runtime if requested.

In [ ]:
# Google Colab setup (run once)
!apt-get update -qq && apt-get install -y openjdk-11-jdk-headless -qq > /dev/null
!pip install -q pyspark==3.5.1
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"

## Step 2 — Create a Spark session

In [ ]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StringType
from pathlib import Path
import time, shutil
spark = (SparkSession.builder.master("local[*]")
         .appName("Day8TwitterHashtagStreaming")
         .config("spark.sql.shuffle.partitions", "2")
         .enableHiveSupport().getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print("Spark version:", spark.version)

## Step 3 — Simulate incoming Twitter messages
The rate-micro-batch source produces **exactly 25 input rows per micro-batch**. Each row is mapped to a sample tweet; this simulates a stream, not the real Twitter/X API. A production connection would require authorized API access or a separate ingestion connector.

In [ ]:
sample_tweets = [
 "Learning #PySpark and #BigData today!",
 "Working with #SparkStreaming and #PySpark",
 "Data science loves #Python #BigData",
 "#AI and #MachineLearning are trending",
 "Exploring #PySpark for #AI projects",
 "Streaming analysis with #SparkStreaming",
 "#Python makes #DataScience fun",
 "Real-time #BigData with #PySpark",
 "New trends in #AI and #DataScience",
 "Try #MachineLearning using #Python"
]
# Rate-micro-batch is supported in Spark 3.5.x; each batch contains 25 tweets.
incoming = (spark.readStream.format("rate-micro-batch")
            .option("rowsPerBatch", 25).option("numPartitions", 1).load())
lookup = F.array(*[F.lit(x) for x in sample_tweets])
tweets = incoming.select(F.col("value").alias("tweet_id"),
                         F.element_at(lookup, (F.col("value") % len(sample_tweets) + 1).cast("int")).alias("tweet"),
                         F.col("timestamp").alias("received_at"))
print("Incoming tweet stream created:", tweets.isStreaming)

## Step 4 — Tokenize tweets and extract hashtags
Split on whitespace, remove punctuation after hashtag words, normalize case, and count a hashtag at most once per tweet. Hashtags are extracted using a regex and `array_distinct`, so repeated occurrences in one tweet do not inflate tweet counts.

In [ ]:
hashtags = (tweets
    .withColumn("hashtag_array", F.array_distinct(F.expr("transform(regexp_extract_all(tweet, '(?i)#[A-Za-z0-9_]+', 0), x -> lower(x))")))
    .withColumn("hashtag", F.explode("hashtag_array"))
    .select("tweet_id", "tweet", "received_at", "hashtag"))
# A static example to inspect before starting the stream:
example = spark.createDataFrame([(1, "#AI #PySpark #AI")], ["tweet_id", "tweet"])
example.select("tweet", F.expr("array_distinct(transform(regexp_extract_all(tweet, '(?i)#[A-Za-z0-9_]+', 0), x -> lower(x)))").alias("hashtags")).show(truncate=False)

## Step 5 — Save counts to a Spark SQL table after every 25 tweets
The input source guarantees 25 tweets per batch. `foreachBatch` maintains cumulative tweet-hashtag counts, saves them to a managed Spark SQL table at the end of each batch, and prints a ranked table after each 180-second interval. The output table is rebuilt from cumulative counts each batch for this instructional demonstration. **Do not run multiple streaming queries writing to the same table.**

In [ ]:
TABLE_NAME = "day8_hashtag_counts"
CHECKPOINT = "/content/day8_hashtag_checkpoint"
# Optional reset for a new demonstration. Do not reset while a query is running.
spark.sql(f"DROP TABLE IF EXISTS {TABLE_NAME}")
shutil.rmtree(CHECKPOINT, ignore_errors=True)

# Driver-side state is appropriate for this small, local classroom simulation.
# For production, use a durable transactional store and idempotent batch writes.
all_counts = {}  # hashtag -> count of tweets containing hashtag
state = {"last_report": time.monotonic(), "processed_tweets": 0}

def process_25_tweets(batch_df, batch_id):
    batch_df.persist()
    try:
        tweet_count = batch_df.select("tweet_id").distinct().count()
        if tweet_count == 0:
            return
        if tweet_count != 25:
            raise ValueError(f"Expected 25 tweets in batch, received {tweet_count}")
        per_tag = batch_df.groupBy("hashtag").agg(F.countDistinct("tweet_id").alias("batch_count"))
        for row in per_tag.collect():
            all_counts[row["hashtag"]] = all_counts.get(row["hashtag"], 0) + row["batch_count"]
        state["processed_tweets"] += tweet_count
        rows = [(tag, int(count)) for tag, count in sorted(all_counts.items())]
        counts_df = spark.createDataFrame(rows, ["hashtag", "tweet_count"])
        counts_df.write.mode("overwrite").saveAsTable(TABLE_NAME)
        print(f"Batch {batch_id}: saved counts after {state['processed_tweets']} total tweets (+{tweet_count}).")
        if time.monotonic() - state["last_report"] >= 180:
            print("\n=== 3-MINUTE HASHTAG POPULARITY REPORT ===")
            spark.sql(f"SELECT hashtag, tweet_count FROM {TABLE_NAME} ORDER BY tweet_count DESC, hashtag").show(100, truncate=False)
            state["last_report"] = time.monotonic()
    finally:
        batch_df.unpersist()

query = (hashtags.writeStream.foreachBatch(process_25_tweets)
         .option("checkpointLocation", CHECKPOINT)
         .trigger(processingTime="10 seconds").start())
print("Streaming started. Let it run for at least 3 minutes to see the timed report.")

## Step 6 — Inspect the Spark SQL table
Run after the first batch completes. The table is updated every 25 tweets, even before the 3-minute report.

In [ ]:
spark.sql("SELECT hashtag, tweet_count FROM day8_hashtag_counts ORDER BY tweet_count DESC, hashtag").show(30, truncate=False)

## Step 7 — Wait for the three-minute report, then stop
Run the next cell to keep the notebook alive for about 190 seconds. The report prints inside the streaming callback after at least 180 seconds; it may appear slightly later depending on micro-batch timing.

In [ ]:
query.awaitTermination(190)
if query.isActive:
    query.stop()
print("Stopped. Total tweets processed:", state["processed_tweets"])
spark.sql("SELECT hashtag, tweet_count FROM day8_hashtag_counts ORDER BY tweet_count DESC, hashtag").show(30, truncate=False)

## Optional — Listening to actual incoming tweet text via a socket
Use a local TCP text feed supplied by an authorized X/Twitter ingestion process. This is an **alternative** input source; it does not independently connect to the Twitter API. A socket source has no guaranteed 25-row batches; to meet the exact 25-tweet save rule, implement a durable tweet counter/buffer or use the supplied fixed-size simulation above.

In [ ]:
# Optional example (do not run unless a TCP server is listening on port 9999):
# socket_tweets = (spark.readStream.format("socket")
#                  .option("host", "localhost").option("port", 9999).load()
#                  .select(F.col("value").alias("tweet")))
# socket_tweets.printSchema()

## Notes and limitations
- Each simulated record represents a tweet; a tweet containing two different hashtags contributes one count to each hashtag.
- Counts are cumulative from the start of the current demonstration.
- The three-minute report uses elapsed processing time, not a 3-minute event-time window.
- `foreachBatch` Python driver state is not restart-safe; the checkpoint alone does not restore `all_counts`. This is an instructional example, not production exactly-once processing.
- In some Spark versions, `regexp_extract_all` may not be available; this notebook targets PySpark 3.5.1.
- Real X/Twitter ingestion requires API authorization and a connector or upstream producer.